# 04 · Triage and approve

*Part 2, sections 5-6 and Part 3, sections 2 and 6.* With more reviewers and more traffic the manual
loop stops scaling, in four ways: **criteria drift**, **reviewer disagreement**, **duplicate
signal**, and **judge drift**. The bottleneck is reading.

Two jobs take it over, deliberately kept apart so the agent that proposes a test never approves it:

* **triage** reads the new failures since its checkpoint, sends contradictions to a person, and has
  an analyst group the rest into themes with one criterion each, proposing tests
* **approve** checks each proposed test and merges it, or sends it to the human queue

In [ ]:
import _nb as nb

experiment_id = nb.setup()

In [ ]:
import mlflow
from examples import reviewer
from selfimprove import agents, config, improvers, instructions, store, traces

drift = [q for q in open("examples/drift.txt").read().splitlines() if q and not q.startswith("#")]
regular = [q for q in open("examples/questions.txt").read().splitlines() if q and not q.startswith("#")]
for q in drift * 2 + regular:
    agents.run(q, source="production")
print(reviewer.review_all(sample=1.0))

A new kind of question has started arriving: escalations, discounts, roadmap requests. v2 never
mentions `support_process`. Meanwhile, the reviewers grade with slightly different rubrics (maria fails a Terms-of-Service
opener, sam fails answers over 90 words). The simulated reviewers each label different traces, so
the count below is 0 here. With real reviewers labeling the same traces, this is where
contradictions show up, and triage sends them to the human queue:

In [ ]:
disputed = [t for t in traces.production() if len(set(traces.human_labels(t).values())) > 1]
print(len(disputed), "traces where reviewers disagree")
for t in traces.production()[:40]:
    labels = traces.human_labels(t)
    if labels:
        print(f"{str(labels):<32} {traces.question_of(t)[:60]}")

## 1. What the analyst reads

The failure report is the signal, compacted: input, output, every failing verdict with its source
and rationale, the tool calls, human failures first. Holdout cases are filtered out before
anything upstream of the gate sees them.

In [ ]:
holdout = store.holdout_questions()
visible = [t for t in traces.production() if not store.is_holdout(traces.question_of(t), holdout)]
report = improvers.failure_report(visible)
print(len(report), "failing traces in the report")
report[0]

**Under the hood:** `improvers.failure_report()` only reads. For each trace it takes the
failing assessments from `trace.info.assessments` (name, `source_type:source_id`, rationale), the
expectation, and the TOOL spans from `trace.search_spans(span_type=SpanType.TOOL)`. The holdout filter
reads the dataset records' `split` tags.

## 2. Triage

`triage()` is the scheduled job from Part 3, section 2. It runs as an MLflow run, and logs its
proposal on that run (`proposal.json`) for the approver. With credentials, the analyst
(`improvers.analyze`, on the judge model) names the themes; without, the first failing verifier does.

In [ ]:
from selfimprove.workflows.triage import triage

result = triage(force=True)
result.metrics

**Under the hood:** every workflow runs inside the harness's `job()` helper:
`mlflow.start_run(run_name="triage <time>")`, tag `si.job=triage`, and the arguments as params.
Triage reads production traces newer than the experiment tag `si.checkpoint.triage`, writes the
proposal with `mlflow.log_dict(..., "proposal.json")` (plus `analysis.json` when the analyst ran),
tags the run `si.proposal=pending`, and logs its counts as metrics with `si.outcome` and `si.message`
tags. Then it moves the checkpoint with `mlflow.set_experiment_tag()`. Contradictions and missing
expectations go to `.selfimprove/notebooks/human_queue.jsonl`, a plain file, not MLflow.

In [ ]:
run = mlflow.get_run(result.data["proposal_run"])
print(run.info.run_name)
print("tags     ", {k: v[:50] for k, v in run.data.tags.items() if k.startswith("si.")})
print("params   ", run.data.params)
print("artifacts", [f.path for f in mlflow.MlflowClient().list_artifacts(run.info.run_id)])
print("experiment tag si.checkpoint.triage =", mlflow.get_experiment(experiment_id).tags.get("si.checkpoint.triage"))

In [ ]:
proposal = mlflow.artifacts.load_dict(f"runs:/{result.data['proposal_run']}/proposal.json")
for th in proposal["themes"]:
    print(f"{th['slug']:<28} x{th['count']:<3} {'(reviewer noise) ' if th['is_reviewer_noise'] else ''}{th['criterion'][:90]}")
print()
for r in proposal["records"][:6]:
    print(f"{r['tags'].get('theme', ''):<28} {r['inputs']['question'][:60]}")

## 3. Approve

A proposed test joins the dataset only if it **reproduces** the failure it came from, is **not a
near duplicate**, stays **under the theme cap** (`SI_TESTS_PER_THEME_CAP`), and passes a
**reviewer agent** (a separate prompt from the analyst). Anything else goes to the human queue.
Real themes with an approved test become registered `theme_<slug>` judges: the verifier grows
with the evidence.

In [ ]:
from selfimprove.workflows.approve import approve

approve().metrics

**Under the hood:** `approve()` is its own run (`si.job=approve`). It finds work with
``mlflow.search_runs(filter_string="tags.`si.job` = 'triage' AND tags.`si.proposal` = 'pending'")``,
reads each proposal with `mlflow.artifacts.load_dict("runs:/<id>/proposal.json")`, merges the approved
records with `merge_records()`, and marks the triage run `si.proposal=done` and
`si.approved_by_run=<approve run id>`. It logs every decision as `verdicts.json` and tags the dataset
with `set_dataset_tags()`. With credentials, each real theme with an approved test becomes
`Guidelines(name="theme_<slug>", ...).register()`, a judge in the experiment's scorer registry.
Flagged tests go to the queue file.

In [ ]:
for r in mlflow.search_runs(filter_string="tags.`si.job` = 'triage'", output_format="list"):
    print(f"triage run {r.info.run_id[:8]}  si.proposal={r.data.tags.get('si.proposal')}  "
          f"si.approved_by_run={r.data.tags.get('si.approved_by_run', '')[:8]}")
approve_run = mlflow.search_runs(filter_string="tags.`si.job` = 'approve'", order_by=["start_time DESC"],
                                 max_results=1, output_format="list")[0]
for v in mlflow.artifacts.load_dict(f"runs:/{approve_run.info.run_id}/verdicts.json")["verdicts"][:4]:
    print(f"approved={v['approved']!s:<6} {v['input'][:48]:<49} {'; '.join(v['reasons'])[:60]}")
from mlflow.genai.datasets import get_dataset
print("dataset tags:", {k: v for k, v in get_dataset(name=config.settings.dataset_name).tags.items()
                        if not k.startswith("mlflow.")})

In [ ]:
for item in store.queue_items():
    print(f"{item['id']}  {item['kind']:<19} {item['reason'][:90]}")

## 4. What a person still owns

The queue: contradictions, failures with no expected answer yet, flagged tests. Resolving an item
is a decision with a record (`selfimprove queue accept|edit|reject <id>` from a terminal). An
edited test counts toward the fourth baseline number below.

In [ ]:
flagged = [i for i in store.queue_items() if i["kind"] == "flagged_test"]
if flagged:
    print(store.resolve(flagged[0]["id"], "edited", note="tightened the expected answer")["status"])
print(store.summary())

**Under the hood:** the queue itself is the JSON-lines file. Resolving an item as `accepted` or
`edited` appends the decision there and merges the record into the MLflow dataset with
`merge_records()`, tagged `source=queue`, `approved_by=human` and `human_edited`.

In [ ]:
df = get_dataset(name=config.settings.dataset_name).to_df()
for _, r in df[df["tags"].map(lambda t: t.get("source") == "queue")].iterrows():
    print(r["inputs"]["question"][:55], {k: r["tags"].get(k) for k in ("source", "approved_by", "human_edited", "split")})

## 5. The four baseline numbers

*Part 2, section 7 and Part 3, section 1.* Traces in, themes out, tests generated, and tests a person
edited before merging. They say how much more of the loop to automate: many edits means the
criteria or the judge do not match the reviewers yet, so fix the verifier first.

In [ ]:
from selfimprove.workflows.report import report

print(report())

**Under the hood:** `report()` reads the loop's own history from MLflow. The triage and approve
rows come from ``mlflow.search_runs(filter_string="tags.`si.job` = 'triage'")`` (and `'approve'`), using
each run's metrics and its `si.approved_by_run` tag. Judge agreement is the experiment tag
`si.judge_agreement`. The queue counts come from the queue file.

**Next:** [05 · Verifier and optimizer](05_verifier_and_optimizer.ipynb).